# Joint parameter grid search — training partition only

Test **all 1,248 feasible combinations** jointly, with `trading_months <= formation_months`.
Formation: 3, 6, 12, 18 months; trading: 2, 3, 6, 12 months; return correlation: 0.8, 0.9;
Engle–Granger significance: 0.01, 0.05; entry: 1.5, 2.0, 2.5; exit: 0, 0.5;
stop: 3.0, 3.5, 4.0, 4.5.

All strategies trade from **2016-07-01** through the session preceding **2023-08-25**.
The last three years remain excluded at price ingestion. Trading windows are contiguous,
nonoverlapping, and the final window is shortened to the common end date.

The grid includes an **18-month formation period**. With data beginning in January 2015, that setting needs January 2015–June 2016 to select pairs before trading can begin in **July 2016**. I used July 2016 as the common trading start for every combination so performance is compared over identical market dates

The engine preserves notebook 03's log-price OLS models, daily log-return correlation
(strictly greater than the threshold), directional EG tests, next-close execution,
fixed holdings, equal allocation across selected pairs, proportional costs, and
stop blacklist until the next window. EG uses the returned **1%/5% critical value**,
consistent with the original 5% rule, rather than substituting an ordinary ADF test.

Formation diagnostics are cached by actual date interval. Each unit-capital pair
simulation is reused across correlation/EG settings with the same selected direction.
Capital scaling is exact for this engine's proportional costs and fractional holdings.
Two window schedules run concurrently by default. Completed combinations are skipped
when resuming; interrupted combinations are rebuilt.

Transaction costs: **5 bps per leg at entry and exit**. The companion `_zero_cost` notebook runs the same strategy with no transaction costs and separate outputs.


In [ ]:
from pathlib import Path
import sys
import time
import pandas as pd
from IPython.display import display

PROJECT_FOLDER = Path.cwd()
if PROJECT_FOLDER.name == "notebooks":
    PROJECT_FOLDER = PROJECT_FOLDER.parent
assert (PROJECT_FOLDER / "src" / "joint_grid_search.py").is_file()
sys.path.insert(0, str(PROJECT_FOLDER / "src"))
from joint_grid_search import GridSearch, parameter_grid, PARAMETER_NAMES, Z_GRID

MAX_WORKERS = 2  # Increase only after checking CPU and memory usage.
TRANSACTION_COST_BPS = 5.0  # Per leg, at both entry and exit; same as notebook 03.
INITIAL_CAPITAL = 100_000.0
OUTPUT_ROOT = PROJECT_FOLDER / "outputs" / "joint_grid_search"
parameters = pd.DataFrame(parameter_grid())
assert len(parameters) == 1248
assert (parameters.trading_months <= parameters.formation_months).all()
display(parameters)
print(f"{len(parameters):,} combinations; {len(Z_GRID)} trading-rule combinations per selection setting")

## Load training data and identify the resumable run

The output folder is identified by input file contents, engine source, package versions,
costs, and initial capital. Changing any of these creates a separate run, preventing stale
cache reuse. Loading and hashing read local files; only pre-holdout price observations
are interpreted or used. Do not run two notebook kernels into the same run folder at once.

In [ ]:
search = GridSearch(PROJECT_FOLDER, output=OUTPUT_ROOT, cost_bps=TRANSACTION_COST_BPS,
                    initial_capital=INITIAL_CAPITAL).load_data()
print(f"Output folder: {search.folder}")
print(f"Training sessions: {len(search.prices):,}; stocks: {len(search.prices.columns):,}")
display(search.summary()[list(PARAMETER_NAMES) + ["Status"]].head())

## Optional formation benchmark

This measures the first formation interval for representative 3- and 18-month windows.
It populates reusable caches and reports candidates requiring directional EG fits.
It is not an extrapolated full-search estimate: later candidate counts and simulation
work can differ substantially. Skip this cell to proceed directly to the search.

In [ ]:
benchmark_rows = []
for formation_months in (3, 18):
    window = search.schedule(formation_months, 2).iloc[0]
    key = f"{window['Formation Start']:%Y%m%d}_{window['Formation End Exclusive']:%Y%m%d}"
    cache_hit = (search.cache_folder / f"{key}.pkl").exists()
    started = time.perf_counter()
    cached = search.formation(window)
    benchmark_rows.append({"formation_months": formation_months, "Cache Hit": cache_hit,
        "Seconds": time.perf_counter() - started,
        "Candidate Pairs": int(cached['comparisons']['Screen Passed'].sum()),
        "Directional Fits": len(cached['directions'])})
display(pd.DataFrame(benchmark_rows))

## Run or resume every feasible combination

This cell performs the full search and displays **one row for each of the 1,248 combinations**,
sorted by full-period Sharpe ratio. It saves each completed combination before proceeding.
The master summary is refreshed after each window-schedule batch. Failed/pending combinations
remain visible, including no-trade combinations. Inspect failures and rerun this cell to resume.

Each parameter folder contains `parameters.json`, `windows.csv`, `selected_pairs.csv`,
`trades.csv`, `portfolio_daily.csv`, `pair_daily.csv.gz`, `window_summary.csv`,
`summary.csv`, and `status.json`. Trades include signal/execution dates and z-scores,
entry/exit prices, signed holdings, hedge ratio, costs, realized P&L, holding duration,
and exit reason. The compressed daily audit records every selected pair's holdings
and account path, including pairs that never enter. Shared formation CSVs record
eligibility, correlations, both EG directions, and rejected-fit diagnostics.

`grid_summary.csv` contains the complete performance table. Runtime is reported for
the shared four-selection trading-rule batch, not as independent per-strategy timing.
Results use zero cash interest and the original engine's cost assumptions; borrow fees,
market impact, and integer-share constraints are not modeled. The gross-equity column
is the existing add-back-of-costs diagnostic, not a separately compounded costless run.

Sharpe is `(Annualized Return - Annualized Risk-Free Return) / Annualized Volatility`. The benchmark uses historical [FRED DGS3MO](https://fred.stlouisfed.org/series/DGS3MO) investment-basis three-month Treasury yields from `data/risk_free/DGS3MO.csv`. On each trading session, use the latest observation strictly before that date (at most seven calendar days old), accrue `(1 + yield / 100) ** (1 / 252) - 1`, compound over the exact evaluation dates, and annualize using 252 sessions/year. This is a rolling-bill yield proxy, not a Treasury total-return index. Window and full-period rates are calculated separately and exported as `Annualized Risk-Free Return`. Missing coverage halts execution; delete the CSV and restart the kernel to refresh. The Treasury data and benchmark implementation are fingerprinted, so old zero-risk-free results are not resumed.


In [ ]:
grid_results = search.run(max_workers=MAX_WORKERS)
with pd.option_context("display.max_rows", len(grid_results), "display.max_columns", None):
    display(grid_results)
print(f"Summary saved to {search.folder / 'grid_summary.csv'}")
print(grid_results["Status"].value_counts().to_string())

## Reload saved results without rerunning

Use training results to investigate stable regions of neighboring parameters. Preserve the
holdout for final evaluation after choosing the strategy; this notebook never tests the holdout.

In [ ]:
saved_results = search.summary()
display(saved_results)
print(f"Complete: {(saved_results['Status'] == 'complete').sum()} / {len(saved_results)}")